[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xomri1/cv-project/blob/main/cv_notebook.ipynb)

# Automatic Object Removal via Click-Based Inpainting
**Computer Vision — Sapienza Università di Roma — A.A. 2025–2026**

Omri Ben Zion & Johannes Erhard

**Pipeline:** User click → SAM (mask) → BLIP-2 (caption) → Improved RePaint SD1.5 → LPIPS / CLIP evaluation

**Code structure (mandatory order):** §1 Imports → §2 Globals → §3 Utils → §4 Data → §5 Network → §6 Train → §7 Evaluation

## Setup

## § 1 · Imports

In [ ]:
%pip uninstall -y -q opencv-python opencv-contrib-python opencv-python-headless
%pip install -q \
    "numpy==1.26.4" \
    "diffusers" \
    "transformers" \
    "accelerate" \
    "bitsandbytes" \
    "datasets==2.19.0" \
    "huggingface_hub" \
    "lpips==0.1.4" \
    "torchmetrics==1.4.0" \
    "opencv-python-headless==4.9.0.80" \
    "scipy==1.13.0" \
    "ipympl==0.9.4" \
    "ipywidgets==8.1.2" \
    "tqdm==4.66.0" \
    "typing_extensions>=4.0"

import os
import time
print("Libraries updated! Restarting kernel in 1 second...", flush=True)
time.sleep(1)
os.kill(os.getpid(), 9)

Libraries updated! Restarting kernel in 1 second...


In [31]:
from google.colab import drive,userdata
drive.mount('/content/drive')
import os
from huggingface_hub import login
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "hf_cache")

_tok = userdata.get('HF_TOKEN')
if _tok:
    login(token=_tok, add_to_git_credential=False)
else:
    print("Set HF_TOKEN in Colab Secrets, or run:  from huggingface_hub import notebook_login; notebook_login()")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Force bitsandbytes to load the CUDA 12.2 binary fallback
os.environ["BNB_CUDA_VERSION"] = "122"

from __future__ import annotations
# ── standard library ──────────────────────────────────────────────────────────
import csv
import gc
import json
import random
import time
from collections import defaultdict
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, cast

# ── numeric / image ───────────────────────────────────────────────────────────
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFilter, ImageFont

# ── visualization ─────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import ipywidgets as widgets

# ── diffusion models ──────────────────────────────────────────────────────────
from diffusers import DDPMScheduler, DiffusionPipeline, StableDiffusionInpaintPipeline
from diffusers.models import AutoencoderKL, UNet2DConditionModel
from diffusers.schedulers.scheduling_utils import SchedulerMixin
try:
    from diffusers.utils.torch_utils import randn_tensor
except ImportError:
    from diffusers.utils import randn_tensor

# ── transformers (SAM, BLIP-2, CLIP) ─────────────────────────────────────────
from transformers import (
    BitsAndBytesConfig,
    Blip2ForConditionalGeneration,
    Blip2Processor,
    CLIPModel,
    CLIPProcessor,
    CLIPTextModel,
    CLIPTokenizer,
    SamModel,
    SamProcessor,
)
from transformers.image_processing_utils import BaseImageProcessor

# ── datasets ──────────────────────────────────────────────────────────────────
from datasets import load_dataset

# ── metrics ───────────────────────────────────────────────────────────────────
import lpips as lpips_lib

# ── utilities ─────────────────────────────────────────────────────────────────
from tqdm.notebook import tqdm

# ── typing override compat ────────────────────────────────────────────────────
try:
    from typing import override
except ImportError:
    try:
        from typing_extensions import override
    except ImportError:
        def override(f):  # type: ignore[misc]
            return f

print("Imports OK — PyTorch", torch.__version__, "| CUDA", torch.cuda.is_available())

This can be used to load a bitsandbytes version built with a CUDA version that is different from the PyTorch CUDA version.
If this was unintended set the BNB_CUDA_VERSION variable to an empty string: export BNB_CUDA_VERSION=

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Imports OK — PyTorch 2.12.0+cu130 | CUDA True


## § 2 · Globals

In [4]:
# ── hardware ──────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── image / inference ─────────────────────────────────────────────────────────
IMAGE_SIZE = 512
SEEDS = [42, 123, 7]
NUM_INFERENCE_STEPS = 50          # RePaint total denoising steps
GUIDANCE_SCALE_START = 5.0        # CFG ramp — low end
GUIDANCE_SCALE_END   = 9.0        # CFG ramp — high end
RESAMPLE_JUMP_LENGTH = 5          # RePaint jump size j
RESAMPLE_TIMES       = 3          # RePaint resample count r
RESAMPLE_WINDOW      = (0.3, 0.7) # fraction of denoising progress (0=noisy, 1=clean)
MASK_DILATION_PX     = 8          # max pixel dilation at t=0
MASK_BLUR_SIGMA      = 4.0        # Gaussian blur sigma for boundary smoothing
SDEDIT_STRENGTH      = 0.3        # SDEdit refinement noise fraction

# ── dataset ───────────────────────────────────────────────────────────────────
DATASET_MODE     = "coco"        # "coco" or "demo"
COCO_SUBSET_SIZE = 100
MASK_AREA_RANGE  = (0.05, 0.25)  # fraction of image pixels
ABLATION_SUBSET  = 30            # samples used for ablation study

# ── prompts ───────────────────────────────────────────────────────────────────
NEG_PROMPT = (
    "blurry, distorted, low quality, watermark, text, deformed, "
    "floating objects, impossible physics, artifacts"
)

# ── paths (Drive) ─────────────────────────────────────────────────────────────
DRIVE_ROOT    = Path("/content/drive/MyDrive/cv_project")
RESULTS_DIR   = DRIVE_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "qualitative").mkdir(exist_ok=True)

INDICES_FILE  = RESULTS_DIR / "coco_subset_indices.json"
QUANT_CSV     = RESULTS_DIR / "coco_quantitative.csv"
ABLATION_CSV  = RESULTS_DIR / "ablation_table.csv"
QUAL_DIR      = RESULTS_DIR / "qualitative"

# ── model IDs (with fallback order) ──────────────────────────────────────────
# Day-1 check: try SD_BASE_IDS in order until one loads cleanly.
SD_INPAINT_IDS = [
    "runwayml/stable-diffusion-inpainting",     # primary
    "botp/stable-diffusion-v1-5-inpainting",    # mirror
]
SD_BASE_IDS = [
    "runwayml/stable-diffusion-v1-5",           # primary (for RePaint)
    "SG161222/Realistic_Vision_V1.4",           # backup
]
SAM_ID   = "facebook/sam-vit-base"
BLIP2_ID = "Salesforce/blip2-opt-2.7b"
CLIP_ID  = "openai/clip-vit-base-patch32"

print("Globals set.")

Device: cuda
Globals set.


## § 3 · Utils

In [5]:
# ── Seed / reproducibility ────────────────────────────────────────────────────

def set_all_seeds(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_generator(seed: int, device: str = DEVICE) -> torch.Generator:
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    return g

In [6]:
# ── VanillaRePaintDiffusionInpaintPipeline ────────────────────────────────────
# Implements the RePaint method (Lugmayr et al., CVPR 2022) for latent
# diffusion. At each reverse step the known region is re-noised from the
# original image, forcing the model to harmonize the boundary.

@dataclass
class RePaintPipelineOutput:
    images: list[Image.Image]


class VanillaRePaintDiffusionInpaintPipeline(DiffusionPipeline):

    _optional_components = ["safety_checker", "feature_extractor"]

    def __init__(
        self,
        vae: AutoencoderKL,
        text_encoder: CLIPTextModel,
        tokenizer: CLIPTokenizer,
        unet: UNet2DConditionModel,
        scheduler: SchedulerMixin,
        safety_checker: Any | None = None,
        feature_extractor: BaseImageProcessor | None = None,
    ) -> None:
        super().__init__()
        if isinstance(scheduler, DDPMScheduler):
            ddpm_scheduler = scheduler
        else:
            scheduler_cfg: Any = getattr(scheduler, "config")
            rebuilt = DDPMScheduler.from_config(scheduler_cfg)
            ddpm_scheduler = rebuilt[0] if isinstance(rebuilt, tuple) else rebuilt
            ddpm_scheduler = cast(DDPMScheduler, ddpm_scheduler)
        self.register_modules(
            vae=vae,
            text_encoder=text_encoder,
            tokenizer=tokenizer,
            unet=unet,
            scheduler=ddpm_scheduler,
            safety_checker=safety_checker,
            feature_extractor=feature_extractor,
        )
        self.vae_scale_factor = 2 ** (len(self.vae.config.block_out_channels) - 1)

    def _encode_prompt(
        self,
        prompt: str | list[str],
        device: torch.device,
        num_images_per_prompt: int,
        do_classifier_free_guidance: bool,
        negative_prompt: str | list[str] | None = None,
    ) -> torch.Tensor:
        batch_size = 1 if isinstance(prompt, str) else len(prompt)
        text_inputs = self.tokenizer(
            prompt,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        )
        prompt_embeds = self.text_encoder(text_inputs.input_ids.to(device))[0]
        bs, seq, hid = prompt_embeds.shape
        prompt_embeds = prompt_embeds.repeat(1, num_images_per_prompt, 1).view(
            bs * num_images_per_prompt, seq, hid
        )
        if do_classifier_free_guidance:
            uncond_tokens = (
                [""] * batch_size
                if negative_prompt is None
                else ([negative_prompt] * batch_size if isinstance(negative_prompt, str) else negative_prompt)
            )
            uncond_input = self.tokenizer(
                uncond_tokens,
                padding="max_length",
                max_length=self.tokenizer.model_max_length,
                truncation=True,
                return_tensors="pt",
            )
            neg_embeds = self.text_encoder(uncond_input.input_ids.to(device))[0]
            neg_embeds = neg_embeds.repeat(1, num_images_per_prompt, 1).view(
                bs * num_images_per_prompt, seq, hid
            )
            prompt_embeds = torch.cat([neg_embeds, prompt_embeds])
        return prompt_embeds

    def _encode_image(
        self,
        image: Image.Image | torch.Tensor,
        device: torch.device,
        dtype: torch.dtype,
    ) -> torch.Tensor:
        if isinstance(image, Image.Image):
            img_np = np.array(image.convert("RGB")).astype(np.float32) / 255.0
            img_t = torch.from_numpy((img_np * 2.0) - 1.0).permute(2, 0, 1).unsqueeze(0)
        else:
            img_t = image
        img_t = img_t.to(device=device, dtype=dtype)
        latents = self.vae.encode(img_t).latent_dist.sample() * self.vae.config.scaling_factor
        return latents

    def _prepare_mask(
        self,
        mask: Image.Image | torch.Tensor,
        height: int,
        width: int,
        device: torch.device,
        dtype: torch.dtype,
    ) -> torch.Tensor:
        if isinstance(mask, Image.Image):
            m_np = np.array(mask.convert("L")).astype(np.float32) / 255.0
            mask = torch.from_numpy(m_np).unsqueeze(0).unsqueeze(0)
        if mask.ndim == 2:
            mask = mask.unsqueeze(0).unsqueeze(0)
        elif mask.ndim == 3:
            mask = mask.unsqueeze(0)
        lh = height // self.vae_scale_factor
        lw = width // self.vae_scale_factor
        mask = F.interpolate(mask.float(), size=(lh, lw), mode="nearest")
        return (mask > 0.5).float().to(device=device, dtype=dtype)

    def _prepare_original_and_mask(
        self,
        image: Image.Image | torch.Tensor,
        mask_image: Image.Image | torch.Tensor,
        width: int,
        height: int,
    ) -> tuple[Image.Image, Image.Image]:
        if isinstance(image, Image.Image):
            original_pil = image.convert("RGB").resize((width, height), Image.Resampling.LANCZOS)
        else:
            pix = (image[0].cpu().permute(1, 2, 0).float().numpy() + 1) / 2 * 255
            original_pil = Image.fromarray(pix.clip(0, 255).astype(np.uint8))
        if isinstance(mask_image, Image.Image):
            mask_pil = mask_image.convert("L").resize((width, height), Image.Resampling.NEAREST)
        else:
            m_np = mask_image.squeeze().cpu().numpy()
            mask_pil = Image.fromarray((m_np * 255).astype(np.uint8)).resize(
                (width, height), Image.Resampling.NEAREST
            )
        return original_pil, mask_pil

    def _get_repaint_schedule(
        self,
        num_inference_steps: int,
        jump_length: int,
        jump_n_sample: int,
    ) -> list[int]:
        jumps: dict[int, int] = {}
        for j in range(0, num_inference_steps - jump_length, jump_length):
            jumps[j] = jump_n_sample - 1
        t = num_inference_steps
        ts: list[int] = []
        while t >= 1:
            t -= 1
            ts.append(t)
            if jumps.get(t, 0) > 0:
                jumps[t] -= 1
                for _ in range(jump_length):
                    t += 1
                    ts.append(t)
        ts.append(-1)
        return ts

    def _sample_step_noise(self, shape, generator, device, dtype) -> torch.Tensor:
        return randn_tensor(shape, generator=generator, device=device, dtype=dtype)

    def _effective_guidance(self, guidance_scale, do_cfg, t_cur, num_inference_steps) -> float:
        return guidance_scale

    def _build_step_mask(self, mask_latents, t_cur, num_inference_steps) -> torch.Tensor:
        return mask_latents

    def _pre_inject_known_region(self, latents, image_latents, mask_step, timestep, step_noise):
        x_known_t = self.scheduler.add_noise(image_latents, step_noise, timestep)
        return mask_step * latents + (1 - mask_step) * x_known_t

    def _reverse_step(self, latents, timestep, prompt_embeds, do_cfg, effective_guidance,
                      generator, step_noise) -> torch.Tensor:
        latent_input = torch.cat([latents] * 2) if do_cfg else latents
        latent_input = self.scheduler.scale_model_input(latent_input, timestep)
        noise_pred = self.unet(latent_input, timestep, encoder_hidden_states=prompt_embeds).sample
        if do_cfg:
            noise_uncond, noise_text = noise_pred.chunk(2)
            noise_pred = noise_uncond + effective_guidance * (noise_text - noise_uncond)
        return self.scheduler.step(noise_pred, timestep, latents, generator=generator).prev_sample

    def _post_reverse_composite(self, latents, image_latents, mask_step, t_cur,
                                 timesteps_array, num_inference_steps, step_noise,
                                 generator, device, dtype) -> torch.Tensor:
        return latents

    def _forward_step(self, latents, t_last, t_cur, timesteps_array, alphas_cumprod,
                       num_inference_steps, generator, device, dtype) -> torch.Tensor:
        if t_cur >= num_inference_steps:
            return randn_tensor(latents.shape, generator=generator, device=device, dtype=dtype)
        src_idx = num_inference_steps - 1 - t_last
        tgt_idx = num_inference_steps - 1 - t_cur
        alpha_bar_src = alphas_cumprod[timesteps_array[src_idx]]
        alpha_bar_tgt = alphas_cumprod[timesteps_array[tgt_idx]]
        ratio = (alpha_bar_tgt / alpha_bar_src).clamp(min=0.0)
        noise = randn_tensor(latents.shape, generator=generator, device=device, dtype=dtype)
        return torch.sqrt(ratio) * latents + torch.sqrt(1.0 - ratio) * noise

    @torch.no_grad()
    def __call__(
        self,
        prompt: str | list[str],
        image: Image.Image | torch.Tensor,
        mask_image: Image.Image | torch.Tensor,
        height: int = IMAGE_SIZE,
        width: int = IMAGE_SIZE,
        num_inference_steps: int = NUM_INFERENCE_STEPS,
        jump_length: int = RESAMPLE_JUMP_LENGTH,
        jump_n_sample: int = RESAMPLE_TIMES,
        guidance_scale: float = 7.5,
        negative_prompt: str | list[str] | None = None,
        num_images_per_prompt: int = 1,
        generator: torch.Generator | None = None,
        callback: Callable[[int, int, torch.Tensor], None] | None = None,
        callback_steps: int = 1,
    ) -> RePaintPipelineOutput:
        device = self._execution_device
        dtype = self.unet.dtype
        do_cfg = guidance_scale > 1.0
        batch_size = 1 if isinstance(prompt, str) else len(prompt)

        prompt_embeds = self._encode_prompt(prompt, device, num_images_per_prompt, do_cfg, negative_prompt)
        image_latents = self._encode_image(image, device, dtype).repeat(num_images_per_prompt, 1, 1, 1)
        mask_latents = self._prepare_mask(mask_image, height, width, device, dtype).repeat(
            batch_size * num_images_per_prompt, 1, 1, 1
        )

        self.scheduler.set_timesteps(num_inference_steps, device=device)
        timesteps_array = self.scheduler.timesteps
        alphas_cumprod = self.scheduler.alphas_cumprod.to(device=device, dtype=dtype)
        schedule = self._get_repaint_schedule(num_inference_steps, jump_length, jump_n_sample)

        latent_shape = (
            batch_size * num_images_per_prompt,
            self.unet.config.in_channels,
            height // self.vae_scale_factor,
            width // self.vae_scale_factor,
        )
        latents = randn_tensor(latent_shape, generator=generator, device=device, dtype=dtype)
        reverse_step_count = 0

        for i in range(len(schedule) - 1):
            t_last, t_cur = schedule[i], schedule[i + 1]
            if t_cur < t_last:
                ts_idx = 0 if t_last >= num_inference_steps else num_inference_steps - 1 - t_last
                timestep = timesteps_array[ts_idx]
                mask_step = self._build_step_mask(mask_latents, t_cur, num_inference_steps)
                eff_g = self._effective_guidance(guidance_scale, do_cfg, t_cur, num_inference_steps)
                step_noise = self._sample_step_noise(tuple(latents.shape), generator, device, dtype)
                latents = self._pre_inject_known_region(latents, image_latents, mask_step, timestep, step_noise)
                latents = self._reverse_step(latents, timestep, prompt_embeds, do_cfg, eff_g, generator, step_noise)
                latents = self._post_reverse_composite(
                    latents, image_latents, mask_step, t_cur, timesteps_array,
                    num_inference_steps, step_noise, generator, device, dtype
                )
                reverse_step_count += 1
                if callback and reverse_step_count % callback_steps == 0:
                    callback(reverse_step_count, timestep, latents)
            else:
                latents = self._forward_step(
                    latents, t_last, t_cur, timesteps_array, alphas_cumprod,
                    num_inference_steps, generator, device, dtype
                )

        latents = latents / self.vae.config.scaling_factor
        decoded = self.vae.decode(latents).sample
        decoded = (decoded / 2 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).float().numpy()
        images = [Image.fromarray((img * 255).round().astype(np.uint8)) for img in decoded]

        original_pil, mask_pil = self._prepare_original_and_mask(image, mask_image, width, height)
        images = [Image.composite(gen, original_pil, mask_pil) for gen in images]
        return RePaintPipelineOutput(images=images)


print("VanillaRePaintDiffusionInpaintPipeline defined.")

VanillaRePaintDiffusionInpaintPipeline defined.


In [7]:
# ── ImprovedRePaintDiffusionInpaintPipeline ───────────────────────────────────
# Adds 8 quality improvements on top of the vanilla RePaint pipeline:
# (a) Coherent noise       (e) Laplacian pyramid blending
# (b) Selective resampling (f) SDEdit refinement pass
# (c) Guidance ramp        (g) Deterministic VAE encoding
# (d) Progressive mask     (h) Default negative prompt


@dataclass(frozen=True)
class _ImprovedCallConfig:
    mask_blur_radius:   int   = 1
    max_mask_dilation:  int   = 1
    pixel_blend_px:     int   = 3
    coherent_noise:     bool  = True
    resample_range:     tuple = (0.2, 0.8)
    guidance_ramp:      bool  = True
    use_laplacian_blend: bool = True
    deterministic_vae:  bool  = True


class ImprovedRePaintDiffusionInpaintPipeline(VanillaRePaintDiffusionInpaintPipeline):

    _active_cfg: _ImprovedCallConfig | None = None

    def _cfg(self) -> _ImprovedCallConfig:
        return self._active_cfg or _ImprovedCallConfig()

    @contextmanager
    def _use_config(self, cfg: _ImprovedCallConfig):
        prev = self._active_cfg
        self._active_cfg = cfg
        try:
            yield
        finally:
            self._active_cfg = prev

    # (g) Deterministic VAE encoding via .mode()
    @override
    def _encode_image(self, image, device, dtype) -> torch.Tensor:
        if isinstance(image, Image.Image):
            img_np = np.array(image.convert("RGB")).astype(np.float32) / 255.0
            image = torch.from_numpy((img_np * 2.0) - 1.0).permute(2, 0, 1).unsqueeze(0)
        image = image.to(device=device, dtype=dtype)
        if self._cfg().deterministic_vae:
            latents = self.vae.encode(image).latent_dist.mode() * self.vae.config.scaling_factor
            return latents
        return super()._encode_image(image, device, dtype)

    @staticmethod
    def _blur_mask(mask: torch.Tensor, blur_radius: int) -> torch.Tensor:
        if blur_radius <= 0:
            return mask
        k = 2 * blur_radius + 1
        sigma = blur_radius / 3.0
        ax = torch.arange(-blur_radius, blur_radius + 1, dtype=mask.dtype, device=mask.device)
        g1d = torch.exp(-0.5 * (ax / sigma) ** 2)
        g1d = g1d / g1d.sum()
        k2d = (g1d[:, None] * g1d[None, :]).view(1, 1, k, k)
        return F.conv2d(mask, k2d, padding=blur_radius).clamp(0.0, 1.0)

    @staticmethod
    def _dilate_mask(mask: torch.Tensor, dilation_px: int) -> torch.Tensor:
        if dilation_px <= 0:
            return mask
        k = 2 * dilation_px + 1
        return F.max_pool2d(mask, kernel_size=k, stride=1, padding=dilation_px)

    @staticmethod
    def _pixel_blend(generated: Image.Image, original: Image.Image,
                     mask: Image.Image, blend_px: int = 12) -> Image.Image:
        if generated.size != original.size:
            generated = generated.resize(original.size, Image.Resampling.LANCZOS)
        mask_f = mask.convert("L")
        if blend_px > 0:
            mask_f = mask_f.filter(ImageFilter.GaussianBlur(radius=blend_px))
        return Image.composite(generated, original, mask_f)

    @staticmethod
    def _multiband_blend(generated: Image.Image, original: Image.Image,
                         mask: Image.Image, num_levels: int = 5) -> Image.Image:
        if generated.size != original.size:
            generated = generated.resize(original.size, Image.Resampling.LANCZOS)
        gen_t  = torch.from_numpy(np.array(generated)).float().permute(2, 0, 1).unsqueeze(0)
        orig_t = torch.from_numpy(np.array(original)).float().permute(2, 0, 1).unsqueeze(0)
        mask_t = (torch.from_numpy(np.array(mask.convert("L"))).float()
                  .unsqueeze(0).unsqueeze(0) / 255.0)

        def _gk(C, sigma=2.0):
            ks = max(int(sigma * 6) | 1, 3)
            h = ks // 2
            x = torch.arange(ks).float() - h
            k1 = torch.exp(-(x**2) / (2 * sigma**2)); k1 /= k1.sum()
            k2 = (k1[None, :] * k1[:, None]).unsqueeze(0).unsqueeze(0).expand(C, 1, -1, -1)
            return k2

        def _blur(t, sigma=2.0):
            C = t.shape[1]; k = _gk(C, sigma); hp = k.shape[-1] // 2
            return F.conv2d(F.pad(t, [hp]*4, mode="reflect"), k, groups=C)

        def _down(t):
            return _blur(t)[:, :, ::2, ::2]

        def _up(t, h, w):
            return F.interpolate(t, size=(h, w), mode="bilinear", align_corners=False)

        def _gpyr(t, L):
            p = [t]
            for _ in range(L - 1):
                if p[-1].shape[2] < 2 or p[-1].shape[3] < 2: break
                p.append(_down(p[-1]))
            return p

        def _lpyr(t, L):
            gp = _gpyr(t, L)
            lp = [gp[i] - _up(gp[i+1], gp[i].shape[2], gp[i].shape[3]) for i in range(len(gp)-1)]
            lp.append(gp[-1])
            return lp

        lp_gen  = _lpyr(gen_t, num_levels)
        lp_orig = _lpyr(orig_t, num_levels)
        gp_mask = _gpyr(mask_t, num_levels)

        blended = [m * lg + (1 - m) * lo for lg, lo, m in zip(lp_gen, lp_orig, gp_mask)]
        result = blended[-1]
        for i in range(len(blended) - 2, -1, -1):
            h, w = blended[i].shape[2], blended[i].shape[3]
            result = _up(result, h, w) + blended[i]

        out = result.squeeze(0).permute(1, 2, 0).clamp(0, 255).byte().numpy()
        return Image.fromarray(out)

    # DDPM step using explicit noise tensor (enables coherent noise compositing)
    def _ddpm_step(self, model_output, timestep, sample, noise):
        t = timestep
        prev_t = t - self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps
        acp   = self.scheduler.alphas_cumprod
        a_t   = acp[t]
        a_tp  = acp[prev_t] if prev_t >= 0 else torch.tensor(1.0, device=a_t.device, dtype=a_t.dtype)
        b_t   = 1 - a_t; b_tp = 1 - a_tp
        ca_t  = a_t / a_tp; cb_t = 1 - ca_t

        pt = self.scheduler.config.prediction_type
        if pt == "epsilon":
            pred_x0 = (sample - b_t**0.5 * model_output) / a_t**0.5
        elif pt == "v_prediction":
            pred_x0 = a_t**0.5 * sample - b_t**0.5 * model_output
        else:
            pred_x0 = model_output

        if getattr(self.scheduler.config, "clip_sample", False):
            cr = getattr(self.scheduler.config, "clip_sample_range", 1.0)
            pred_x0 = pred_x0.clamp(-cr, cr)

        mean = a_tp**0.5 * cb_t / b_t * pred_x0 + ca_t**0.5 * b_tp / b_t * sample
        var  = (b_tp / b_t * cb_t).clamp(min=1e-20)
        prev = mean + var**0.5 * noise if t > 0 else mean
        return prev, pred_x0

    # (b) Selective resampling
    @override
    def _get_repaint_schedule(self, num_inference_steps, jump_length, jump_n_sample) -> list[int]:
        N = num_inference_steps
        rr = self._cfg().resample_range
        lo = int((1.0 - rr[1]) * max(N - 1, 1))
        hi = int((1.0 - rr[0]) * max(N - 1, 1))
        jumps: dict[int, int] = {}
        for j in range(0, N - jump_length, jump_length):
            if lo <= j <= hi:
                jumps[j] = jump_n_sample - 1
        t = N; ts: list[int] = []
        while t >= 1:
            t -= 1; ts.append(t)
            if jumps.get(t, 0) > 0:
                jumps[t] -= 1
                for _ in range(jump_length): t += 1; ts.append(t)
        ts.append(-1)
        return ts

    # (c) Guidance ramp
    @override
    def _effective_guidance(self, guidance_scale, do_cfg, t_cur, num_inference_steps) -> float:
        if not (self._cfg().guidance_ramp and do_cfg):
            return guidance_scale
        prog = 1.0 - (max(t_cur, 0) / max(num_inference_steps - 1, 1))
        lo = max(1.0, guidance_scale * 0.5)
        return lo + (guidance_scale - lo) * prog

    # (d) Progressive mask dilation + blur
    @override
    def _build_step_mask(self, mask_latents, t_cur, num_inference_steps) -> torch.Tensor:
        cfg = self._cfg()
        prog = 1.0 - (max(t_cur, 0) / max(num_inference_steps - 1, 1))
        m = mask_latents
        if cfg.max_mask_dilation > 0:
            dil = int(round(cfg.max_mask_dilation * (1.0 - prog)))
            m = self._dilate_mask(m, dil)
        if cfg.mask_blur_radius > 0 and prog < 0.8:
            blr = max(1, int(round(cfg.mask_blur_radius * (1.0 - prog / 0.8))))
            m = self._blur_mask(m, blr)
        return m

    # (a) Coherent noise reverse step
    @override
    def _reverse_step(self, latents, timestep, prompt_embeds, do_cfg, effective_guidance,
                      generator, step_noise) -> torch.Tensor:
        latent_input = torch.cat([latents] * 2) if do_cfg else latents
        latent_input = self.scheduler.scale_model_input(latent_input, timestep)
        noise_pred = self.unet(latent_input, timestep, encoder_hidden_states=prompt_embeds).sample
        if do_cfg:
            nu, nt = noise_pred.chunk(2)
            noise_pred = nu + effective_guidance * (nt - nu)
        if self._cfg().coherent_noise:
            prev, _ = self._ddpm_step(noise_pred, int(timestep.item()), latents, step_noise)
            return prev
        return self.scheduler.step(noise_pred, timestep, latents, generator=generator).prev_sample

    @override
    def _post_reverse_composite(self, latents, image_latents, mask_step, t_cur,
                                 timesteps_array, num_inference_steps, step_noise,
                                 generator, device, dtype) -> torch.Tensor:
        if t_cur >= 0:
            tgt = timesteps_array[num_inference_steps - 1 - t_cur]
            x_known = self.scheduler.add_noise(image_latents, step_noise, tgt)
        else:
            x_known = image_latents
        return mask_step * latents + (1 - mask_step) * x_known

    @torch.no_grad()
    def _refinement_pass(self, image, prompt_embeds, height, width, strength,
                          num_steps, guidance_scale, do_cfg, generator, device, dtype) -> Image.Image:
        latents = self._encode_image(image, device, dtype)
        self.scheduler.set_timesteps(num_steps, device=device)
        ts = self.scheduler.timesteps
        si = max(int(len(ts) * (1.0 - strength)), 0)
        noise = randn_tensor(latents.shape, generator=generator, device=device, dtype=dtype)
        latents = self.scheduler.add_noise(latents, noise, ts[si])
        for t in ts[si:]:
            li = torch.cat([latents] * 2) if do_cfg else latents
            li = self.scheduler.scale_model_input(li, t)
            np_ = self.unet(li, t, encoder_hidden_states=prompt_embeds).sample
            if do_cfg:
                nu, nt = np_.chunk(2); np_ = nu + guidance_scale * (nt - nu)
            latents = self.scheduler.step(np_, t, latents, generator=generator).prev_sample
        latents = latents / self.vae.config.scaling_factor
        dec = self.vae.decode(latents).sample
        dec = (dec / 2 + 0.5).clamp(0, 1)[0].cpu().permute(1, 2, 0).float().numpy()
        return Image.fromarray((dec * 255).round().astype(np.uint8))

    @override
    @torch.no_grad()
    def __call__(
        self,
        prompt: str | list[str],
        image: Image.Image | torch.Tensor,
        mask_image: Image.Image | torch.Tensor,
        height: int = IMAGE_SIZE,
        width: int = IMAGE_SIZE,
        num_inference_steps: int = NUM_INFERENCE_STEPS,
        jump_length: int = RESAMPLE_JUMP_LENGTH,
        jump_n_sample: int = RESAMPLE_TIMES,
        guidance_scale: float = 7.5,
        negative_prompt: str | list[str] | None = None,
        num_images_per_prompt: int = 1,
        generator: torch.Generator | None = None,
        callback: Callable[[int, int, torch.Tensor], None] | None = None,
        callback_steps: int = 1,
        # ---- improvement knobs ----
        mask_blur_radius:   int   = 1,
        max_mask_dilation:  int   = 1,
        pixel_blend_px:     int   = 3,
        coherent_noise:     bool  = True,
        resample_range:     tuple = (0.2, 0.8),
        guidance_ramp:      bool  = True,
        use_laplacian_blend: bool = True,
        deterministic_vae:  bool  = True,
        refinement_strength: float = 0.3,
        refinement_steps:   int   = 15,
    ) -> RePaintPipelineOutput:
        device = self._execution_device
        dtype  = self.unet.dtype
        do_cfg = guidance_scale > 1.0

        # (h) Default negative prompt
        if negative_prompt is None and do_cfg:
            negative_prompt = NEG_PROMPT

        original_pil, mask_pil = self._prepare_original_and_mask(image, mask_image, width, height)
        prompt_embeds = self._encode_prompt(prompt, device, num_images_per_prompt, do_cfg, negative_prompt)

        cfg = _ImprovedCallConfig(
            mask_blur_radius=mask_blur_radius,
            max_mask_dilation=max_mask_dilation,
            pixel_blend_px=pixel_blend_px,
            coherent_noise=coherent_noise,
            resample_range=resample_range,
            guidance_ramp=guidance_ramp,
            use_laplacian_blend=use_laplacian_blend,
            deterministic_vae=deterministic_vae,
        )

        with self._use_config(cfg):
            base = super().__call__(
                prompt=prompt, image=image, mask_image=mask_image,
                height=height, width=width,
                num_inference_steps=num_inference_steps,
                jump_length=jump_length, jump_n_sample=jump_n_sample,
                guidance_scale=guidance_scale, negative_prompt=negative_prompt,
                num_images_per_prompt=num_images_per_prompt,
                generator=generator, callback=callback, callback_steps=callback_steps,
            )

            # (e) Laplacian pyramid blending  /  pixel feather
            images = (
                [self._multiband_blend(img, original_pil, mask_pil) for img in base.images]
                if cfg.use_laplacian_blend else
                [self._pixel_blend(img, original_pil, mask_pil, cfg.pixel_blend_px) for img in base.images]
            )

            # (f) SDEdit refinement pass
            if refinement_strength > 0.0 and refinement_steps > 0:
                refined = []
                for img in images:
                    ref = self._refinement_pass(
                        img, prompt_embeds, height, width,
                        refinement_strength, refinement_steps,
                        guidance_scale, do_cfg, generator, device, dtype
                    )
                    ref = (self._multiband_blend(ref, original_pil, mask_pil)
                           if cfg.use_laplacian_blend else
                           self._pixel_blend(ref, original_pil, mask_pil, cfg.pixel_blend_px))
                    refined.append(ref)
                images = refined

        # Final hard paste-back preserves non-masked pixels exactly
        images = [Image.composite(gen, original_pil, mask_pil) for gen in images]
        return RePaintPipelineOutput(images=images)


print("ImprovedRePaintDiffusionInpaintPipeline defined.")

ImprovedRePaintDiffusionInpaintPipeline defined.


In [8]:
# ── SAM utilities ─────────────────────────────────────────────────────────────
# sam_model and sam_processor are initialized in §5 Network.

def _postprocess_mask(mask_np: np.ndarray) -> np.ndarray:
    # morphological close (fill holes / jagged edges)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    closed = cv2.morphologyEx((mask_np * 255).astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    return closed


def mask_from_click(image: Image.Image, x: int, y: int) -> Image.Image:
    img_rgb = np.array(image.convert("RGB"))
    inputs = sam_processor(
        images=img_rgb,
        input_points=[[[x, y]]],
        return_tensors="pt",
    ).to(DEVICE)
    with torch.no_grad():
        outputs = sam_model(**inputs)
    masks = sam_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu(),
    )
    iou_scores = outputs.iou_scores[0, 0]   # (3,)
    best_idx   = int(iou_scores.argmax())
    best_mask  = masks[0][0][best_idx].numpy().astype(np.float32)
    mask_img   = _postprocess_mask(best_mask)
    # resize to IMAGE_SIZE
    pil_mask = Image.fromarray(mask_img).resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST)
    return pil_mask


def mask_to_bbox_crop(mask: Image.Image, padding: int = 20) -> tuple[int, int, int, int]:
    m = np.array(mask.convert("L"))
    rows = np.any(m > 127, axis=1); cols = np.any(m > 127, axis=0)
    if not rows.any():
        return (0, 0, m.shape[1], m.shape[0])
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    h, w = m.shape
    return (max(0, int(cmin) - padding), max(0, int(rmin) - padding),
            min(w, int(cmax) + padding), min(h, int(rmax) + padding))

In [9]:
# ── BLIP-2 captioning ─────────────────────────────────────────────────────────
# blip2_model and blip2_processor are initialized in §5 Network.

_BAD_CAPTION_WORDS = frozenset(["black", "square", "hole", "missing", "dark patch", "empty"])
_FALLBACK_CAPTION  = "a high quality photograph of the background scene"


def caption_background(image: Image.Image, mask: Image.Image) -> str:
    # Black out the masked region, then caption the background
    img_np   = np.array(image.convert("RGB"))
    mask_np  = np.array(mask.convert("L"))
    masked   = img_np.copy()
    masked[mask_np > 127] = 0
    masked_pil = Image.fromarray(masked)

    question = "Question: What is visible in the background of this image? Answer:"
    inputs = blip2_processor(images=masked_pil, text=question, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    # Cast float inputs to fp16 to match model weights
    for k in inputs:
        if inputs[k].dtype == torch.float32:
            inputs[k] = inputs[k].half()

    with torch.no_grad():
        ids = blip2_model.generate(**inputs, max_new_tokens=35)

    caption = blip2_processor.decode(ids[0], skip_special_tokens=True).strip()

    # Strip echoed question prefix
    if "Answer:" in caption:
        caption = caption.split("Answer:")[-1].strip()

    # Fallback if captioning the black region
    if any(w in caption.lower() for w in _BAD_CAPTION_WORDS) or not caption:
        return _FALLBACK_CAPTION

    # Truncate to 30 tokens to stay inside CLIP's 77-token limit
    tokens = caption.split()
    if len(tokens) > 30:
        caption = " ".join(tokens[:30])

    return caption

In [10]:
# ── Metrics ───────────────────────────────────────────────────────────────────
# lpips_fn and clip_model/clip_processor initialized in §5 Network.

def compute_lpips(gen: Image.Image, ref: Image.Image) -> float:
    g = TF.to_tensor(gen.convert("RGB")).unsqueeze(0).to(DEVICE) * 2 - 1
    r = TF.to_tensor(ref.convert("RGB")).unsqueeze(0).to(DEVICE) * 2 - 1
    with torch.no_grad():
        return float(lpips_fn(g, r))


def compute_mask_lpips(gen: Image.Image, ref: Image.Image, mask: Image.Image) -> float:
    bbox = mask_to_bbox_crop(mask)
    return compute_lpips(gen.crop(bbox), ref.crop(bbox))


def compute_clip_score(image: Image.Image, prompt: str) -> float:
    inputs = clip_processor(text=[prompt], images=[image], return_tensors="pt", padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = clip_model(**inputs)
    ie = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    te = out.text_embeds  / out.text_embeds.norm(dim=-1, keepdim=True)
    return float((ie * te).sum(dim=-1))

In [11]:
# ── Visualization ─────────────────────────────────────────────────────────────

def visualize_grid(images: list, labels: list, figsize: tuple | None = None) -> None:
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize or (4 * n, 4))
    if n == 1: axes = [axes]
    for ax, img, lbl in zip(axes, images, labels):
        ax.imshow(np.array(img) if not isinstance(img, np.ndarray) else img)
        ax.set_title(lbl, fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()


def make_comparison_grid(
    original: Image.Image,
    mask: Image.Image,
    results: dict,       # {name: Image}
    save_path: Path | None = None,
) -> Image.Image:
    images = [original, mask] + list(results.values())
    labels = ["Original", "Mask"]  + list(results.keys())
    n = len(images); cw = ch = IMAGE_SIZE; lh = 28; pad = 4
    grid = Image.new("RGB", (n * cw + (n - 1) * pad, ch + lh), (255, 255, 255))
    draw = ImageDraw.Draw(grid)
    try: font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 13)
    except: font = ImageFont.load_default()
    for i, (img, lbl) in enumerate(zip(images, labels)):
        x = i * (cw + pad)
        bbox = draw.textbbox((0, 0), lbl, font=font)
        draw.text((x + (cw - (bbox[2] - bbox[0])) // 2, 4), lbl, fill=(0, 0, 0), font=font)
        grid.paste(img.resize((cw, ch), Image.Resampling.LANCZOS), (x, lh))
    if save_path: grid.save(str(save_path))
    return grid

In [12]:
# ── Checkpoint utilities ──────────────────────────────────────────────────────
# Saves rows incrementally after each sample so Colab crashes lose at most one sample.

_QUANT_FIELDS = ["image_id", "category", "seed", "pipeline",
                 "lpips", "mask_lpips", "clip_score", "runtime_sec", "caption"]


def checkpoint_append(rows: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists() or path.stat().st_size == 0
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=_QUANT_FIELDS, extrasaction="ignore")
        if write_header: w.writeheader()
        w.writerows(rows)


def load_done_set(path: Path) -> set[tuple]:
    done: set[tuple] = set()
    if not path.exists(): return done
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            done.add((row["image_id"], row["seed"], row["pipeline"]))
    return done

In [13]:
import shutil
MODELS_CACHE_DIR = DRIVE_ROOT / "models"
MODELS_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def get_model_path_local(model_id: str) -> str:
    """Like get_model_path but copies from Drive to /tmp for fast loading."""
    local_name = model_id.replace("/", "_")
    drive_path = MODELS_CACHE_DIR / local_name
    local_path = Path("/tmp/model_cache") / local_name

    if local_path.exists():
        return str(local_path)  # already copied this session
    if drive_path.exists():
        print(f"Copying {local_name} from Drive to local... ", end="", flush=True)
        shutil.copytree(str(drive_path), str(local_path))
        print("done.")
        return str(local_path)
    return model_id  # fall back to HF Hub download


def get_model_path(model_id: str) -> str:
    """Returns the Drive path if cached, otherwise returns the Hugging Face ID."""
    # Create a safe folder name from the model ID (e.g., 'runwayml/sd-v1-5' -> 'runwayml_sd-v1-5')
    local_name = model_id.replace("/", "_")
    local_path = MODELS_CACHE_DIR / local_name

    if local_path.exists() and any(local_path.iterdir()):
        print(f"Using cached model from Drive: {local_path}")
        return str(local_path)

    print(f"Model {model_id} not in cache. Will download and save to Drive.")
    return model_id

def cache_model_to_drive(model_obj, model_id: str):
    """Saves a downloaded model/pipeline to Drive cache."""
    local_name = model_id.replace("/", "_")
    local_path = MODELS_CACHE_DIR / local_name
    if not local_path.exists():
        print(f"Caching {model_id} to {local_path}...")
        model_obj.save_pretrained(str(local_path))
        print("Done.")

In [14]:
print(f"Checking cache at: {MODELS_CACHE_DIR}")
if MODELS_CACHE_DIR.exists():
    cached_models = [d.name for d in MODELS_CACHE_DIR.iterdir() if d.is_dir()]
    if cached_models:
        print("Successfully cached models:")
        for m in cached_models:
            print(f" - {m}")
    else:
        print("Cache directory is empty.")
else:
    print("Cache directory does not exist yet.")

Checking cache at: /content/drive/MyDrive/cv_project/models
Successfully cached models:
 - runwayml--stable-diffusion-inpainting
 - runwayml--stable-diffusion-v1-5
 - facebook--sam-vit-base
 - runwayml_stable-diffusion-inpainting
 - runwayml_stable-diffusion-v1-5
 - Salesforce_blip2-opt-2.7b
 - facebook_sam-vit-base


## § 4 · Data

In [32]:
# ── COCO subset selection ─────────────────────────────────────────────────────
# Filters COCO 2017 val for medium-size objects (5-25% of image area),
# caps 10 instances per category, saves indices for reproducibility.

def load_coco_subset(
    size: int = COCO_SUBSET_SIZE,
    area_range: tuple = MASK_AREA_RANGE,
    seed: int = 42,
) -> list[dict]:
    print("Loading shunk031/MSCOCO 2017 val")
    ds = load_dataset(
        "shunk031/MSCOCO",
        year=2017,
        coco_task="instances",
        split="validation",
        decode_rle=True,
        trust_remote_code=True,
    )

    rng = random.Random(seed)
    candidates: list[dict] = []

    for idx, item in enumerate(tqdm(ds, desc="Filtering")):
        pil_img = item["image"].convert("RGB")
        W, H    = pil_img.size
        area_px = W * H

        anns = item.get("annotations") or []
        valid = []
        for ann in anns:
            if ann.get("iscrowd", 0):
                continue
            seg = ann.get("segmentation")
            if seg is None:
                continue
            # After decode_rle=True, segmentation is a PIL Image or np.ndarray
            if isinstance(seg, Image.Image):
                seg_np = np.array(seg.convert("L"))
            elif isinstance(seg, np.ndarray):
                seg_np = seg.astype(np.uint8)
            else:
                continue
            frac = float(np.count_nonzero(seg_np)) / area_px
            if area_range[0] <= frac <= area_range[1]:
                cat_name = (ann.get("category") or {}).get("name", "object")
                valid.append({"ann": ann, "frac": frac, "cat": cat_name, "seg": seg_np})

        if not valid:
            continue

        # Pick annotation closest to range midpoint
        mid  = (area_range[0] + area_range[1]) / 2.0
        best = min(valid, key=lambda v: abs(v["frac"] - mid))

        # Build binary mask PIL
        mask_pil = Image.fromarray(
            (best["seg"] > 0).astype(np.uint8) * 255
        ).resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST)

        candidates.append({
            "idx":      idx,
            "image_id": item.get("image_id", idx),
            "image":    pil_img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS),
            "mask":     mask_pil,
            "category": best["cat"],
            "area_frac": best["frac"],
        })

    print(f"Candidates after area filter: {len(candidates)}")

    # Category stratification — cap 10 per category
    by_cat: dict[str, list] = defaultdict(list)
    for c in candidates:
        by_cat[c["category"]].append(c)
    stratified: list[dict] = []
    for cat_items in by_cat.values():
        rng.shuffle(cat_items)
        stratified.extend(cat_items[:10])

    rng.shuffle(stratified)
    subset = stratified[:size]
    print(f"Final subset size: {len(subset)}")
    return subset


def save_subset_indices(subset: list[dict], path: Path = INDICES_FILE) -> None:
    idxs = [{"idx": s["idx"], "image_id": s["image_id"], "category": s["category"]} for s in subset]
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(idxs, f, indent=2)
    print(f"Saved indices → {path}")

In [33]:
# ── Demo image loader (local files or Rome photos) ────────────────────────────

def load_demo_images(folder: str | Path) -> list[dict]:
    folder = Path(folder)
    exts   = {".jpg", ".jpeg", ".png", ".webp"}
    samples = []
    for p in sorted(folder.iterdir()):
        if p.suffix.lower() not in exts:
            continue
        img = Image.open(p).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
        samples.append({"image": img, "path": str(p), "image_id": p.stem})
    print(f"Loaded {len(samples)} demo images from {folder}")
    return samples

## § 5 · Network

In [34]:
def load_vanilla_repaint_pipeline(model_id: str | None = None) -> VanillaRePaintDiffusionInpaintPipeline:
    for mid in ([model_id] if model_id else SD_BASE_IDS):
        try:
            path = get_model_path_local(mid)
            print(f"Loading Vanilla RePaint from {path}")
            pipe = _build_repaint_pipe(mid, VanillaRePaintDiffusionInpaintPipeline)
            try: pipe.enable_xformers_memory_efficient_attention()
            except Exception: pass
            pipe.set_progress_bar_config(disable=True)
            return pipe
        except Exception as e:
            print(f"[WARN] {mid} failed: {e}")
    raise RuntimeError("All SD base checkpoints failed to load.")

def load_improved_repaint_pipeline(model_id: str | None = None) -> ImprovedRePaintDiffusionInpaintPipeline:
    for mid in ([model_id] if model_id else SD_BASE_IDS):
        try:
            path = get_model_path_local(mid)
            print(f"Loading Improved RePaint from {path}")
            pipe = _build_repaint_pipe(mid, ImprovedRePaintDiffusionInpaintPipeline)
            try: pipe.enable_xformers_memory_efficient_attention()
            except Exception: pass
            pipe.set_progress_bar_config(disable=True)
            return pipe
        except Exception as e:
            print(f"[WARN] {mid} failed: {e}")
    raise RuntimeError("All SD base checkpoints failed to load.")

def load_repaint_pipelines_shared(model_id=None):
# Load SD base model ONCE and construct both RePaint pipelines from the
# same component objects (vae / text_encoder / unet / tokenizer).
# Vanilla and Improved share identical weights — loading twice wastes
# ~3.5 GB of VRAM unnecessarily.
  for mid in ([model_id] if model_id else SD_BASE_IDS):
      try:
          print(f"Loading SD base (shared for both RePaint variants): {mid}")
          base = DiffusionPipeline.from_pretrained(
              mid, torch_dtype=torch.float16, safety_checker=None
          )
          shared = dict(
              vae=base.vae,
              text_encoder=base.text_encoder,
              tokenizer=base.tokenizer,
              unet=base.unet,
              scheduler=base.scheduler,
              safety_checker=getattr(base, "safety_checker", None),
              feature_extractor=getattr(base, "feature_extractor", None),
          )
          vanilla  = VanillaRePaintDiffusionInpaintPipeline(**shared).to(DEVICE)
          improved = ImprovedRePaintDiffusionInpaintPipeline(**shared).to(DEVICE)
          del base, shared; gc.collect()
          for p in [vanilla, improved]:
              try: p.enable_xformers_memory_efficient_attention()
              except Exception: pass
              p.set_progress_bar_config(disable=True)
          print(f"Vanilla + Improved RePaint loaded (shared weights): {mid}")
          return vanilla, improved
      except Exception as e:
          print(f"[WARN] {mid} failed: {e}")
  raise RuntimeError("All SD base checkpoints failed to load.")

In [35]:
def load_sam(model_id: str = SAM_ID) -> tuple:
    path = get_model_path_local(model_id)
    print(f"Loading SAM from {path}")
    model = SamModel.from_pretrained(path, torch_dtype=torch.float16).to(DEVICE).eval()
    processor = SamProcessor.from_pretrained(path)
    if path == model_id:
        cache_model_to_drive(model, model_id)
        processor.save_pretrained(str(MODELS_CACHE_DIR / model_id.replace("/", "_")))
    return model, processor

def load_blip2(model_id: str = BLIP2_ID, quant: str = "int8") -> tuple:
    path = get_model_path_local(model_id)
    bnb_cfg = BitsAndBytesConfig(load_in_8bit=True) if quant == "int8" else BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = Blip2ForConditionalGeneration.from_pretrained(path, quantization_config=bnb_cfg, device_map="auto")
    processor = Blip2Processor.from_pretrained(path)
    if path == model_id:
        # Note: Quantized models have specific saving requirements, usually saving the base config/weights works
        model.save_pretrained(str(MODELS_CACHE_DIR / model_id.replace("/", "_")))
        processor.save_pretrained(str(MODELS_CACHE_DIR / model_id.replace("/", "_")))
    return model, processor

def load_sd_inpaint_pipeline(model_id: str | None = None) -> StableDiffusionInpaintPipeline:
    ids = [model_id] if model_id else SD_INPAINT_IDS
    for mid in ids:
        try:
            path = get_model_path_local(mid)
            pipe = StableDiffusionInpaintPipeline.from_pretrained(path, torch_dtype=torch.float16, safety_checker=None).to(DEVICE)
            if path == mid:
                pipe.save_pretrained(str(MODELS_CACHE_DIR / mid.replace("/", "_")))
            return pipe
        except Exception as e:
            print(f"Failed {mid}: {e}")
    raise RuntimeError("Failed to load Inpaint Pipeline")

def _build_repaint_pipe(model_id: str, cls):
    path = get_model_path_local(model_id)
    base = DiffusionPipeline.from_pretrained(path, torch_dtype=torch.float16, safety_checker=None)
    if path == model_id:
        base.save_pretrained(str(MODELS_CACHE_DIR / model_id.replace("/", "_")))
    pipe = cls(vae=base.vae, text_encoder=base.text_encoder, tokenizer=base.tokenizer, unet=base.unet, scheduler=base.scheduler)
    return pipe.to(DEVICE)

In [19]:
# ── Load all models ───────────────────────────────────────────────────────────
# Run this cell once per Colab session.
# Order: SD first (largest) → BLIP-2 → SAM (smallest), to catch OOM early.

print("=" * 60)

sd_inpaint_pipe   = load_sd_inpaint_pipeline()      # baseline
torch.cuda.empty_cache()

vanilla_pipe, improved_pipe = load_repaint_pipelines_shared()
torch.cuda.empty_cache()

blip2_model, blip2_processor = load_blip2()         # BLIP-2 (captioning)
torch.cuda.empty_cache()

sam_model, sam_processor     = load_sam()            # SAM (segmentation)
torch.cuda.empty_cache()

lpips_fn     = lpips_lib.LPIPS(net="alex").to(DEVICE).eval()
clip_model   = CLIPModel.from_pretrained(CLIP_ID).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)

print("=" * 60)
print("All models loaded.")
if torch.cuda.is_available():
    print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB / "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Copying runwayml_stable-diffusion-inpainting from Drive to local... done.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion_inpaint.StableDiffusionInpaintPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Loading SD base (shared for both RePaint variants): runwayml/stable-diffusion-v1-5


...

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

All models loaded.
VRAM used: 9.62 GB / 15.64 GB


In [20]:
# import torch
# from PIL import Image
# import numpy as np

# # Create a dummy test image (solid color) and a simple mask
# test_img = Image.fromarray(np.full((IMAGE_SIZE, IMAGE_SIZE, 3), 128, dtype=np.uint8))
# test_mask = Image.fromarray(np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8))
# # Put a white square in the middle of the mask
# test_mask.paste(255, (200, 200, 312, 312))

# print("Running test inference...")
# try:
#     test_out = run_pipeline(
#         image=test_img,
#         click_or_mask=test_mask,
#         mode="coco",
#         pipeline="improved",
#         verbose=True
#     )
#     print("\nTest Inference Success!")
#     display(test_out['result'])
#     print(f"Pipeline times: {test_out['times']}")
# except Exception as e:
#     print(f"\nTest Inference Failed: {e}")

## § 6 · Train
> *No model training in this project.* This section assembles the full inference pipeline and defines experiment runners.

In [36]:
# ── run_pipeline ─────────────────────────────────────────────────────────────
# Full pipeline: image + (click | mask) → inpainted image.

def run_pipeline(
    image: Image.Image,
    click_or_mask,          # (x, y) tuple for demo mode, PIL Image for COCO mode
    mode: str = DATASET_MODE,
    seed: int = 42,
    pipeline: str = "improved",   # "baseline" | "vanilla" | "improved"
    verbose: bool = False,
) -> dict:
    set_all_seeds(seed)
    gen = make_generator(seed)
    t0  = time.perf_counter()

    # Step 1: mask
    if isinstance(click_or_mask, tuple):
        x, y = click_or_mask
        mask = mask_from_click(image, x, y)
        t_mask = time.perf_counter() - t0
    else:
        mask = click_or_mask.convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST)
        t_mask = 0.0

    # Step 2: caption
    t1     = time.perf_counter()
    prompt = caption_background(image, mask)
    t_cap  = time.perf_counter() - t1
    if verbose:
        print(f"  Caption: {prompt!r}")

    # Step 3: inpaint
    t2 = time.perf_counter()
    if pipeline == "baseline":
        result = sd_inpaint_pipe(
            prompt=prompt, image=image, mask_image=mask, generator=gen
        ).images[0]
    elif pipeline == "vanilla":
        result = vanilla_pipe(
            prompt=prompt, image=image, mask_image=mask, generator=gen
        ).images[0]
    else:
        result = improved_pipe(
            prompt=prompt, image=image, mask_image=mask, generator=gen
        ).images[0]
    t_inpaint = time.perf_counter() - t2
    t_total   = time.perf_counter() - t0

    return {
        "result": result,
        "mask":   mask,
        "prompt": prompt,
        "times": {
            "mask_sec":    round(t_mask,   2),
            "caption_sec": round(t_cap,    2),
            "inpaint_sec": round(t_inpaint, 2),
            "total_sec":   round(t_total,  2),
        },
    }

In [37]:
# ── run_ablation ──────────────────────────────────────────────────────────────
# Runs the improved pipeline with one improvement disabled at a time.

ABLATION_KNOBS = {
    "No Coherent Noise":       {"coherent_noise": False},
    "No Selective Resampling": {"resample_range": (0.0, 1.0)},
    "No Guidance Ramp":        {"guidance_ramp": False},
    "No Mask Softening":       {"mask_blur_radius": 0, "max_mask_dilation": 0},
    "No Laplacian Blending":   {"use_laplacian_blend": False},
    "No Refinement Pass":      {"refinement_strength": 0.0, "refinement_steps": 0},
    "No Det. VAE Encoding":    {"deterministic_vae": False},
    "No Negative Prompt":      {},   # handled below
}


def run_ablation(
    image: Image.Image,
    mask: Image.Image,
    prompt: str,
    seed: int = 42,
) -> dict[str, Image.Image]:
    set_all_seeds(seed)
    results: dict[str, Image.Image] = {}

    for name, knobs in ABLATION_KNOBS.items():
        gen = make_generator(seed)
        neg = "" if name == "No Negative Prompt" else None   # disabled vs. default
        result = improved_pipe(
            prompt=prompt, image=image, mask_image=mask,
            generator=gen, negative_prompt=neg, **knobs
        ).images[0]
        results[name] = result

    return results


def run_multi_seed(
    image: Image.Image,
    mask: Image.Image,
    prompt: str,
    pipeline: str = "improved",
    seeds: list[int] = SEEDS,
) -> list[Image.Image]:
    outputs = []
    for s in seeds:
        res = run_pipeline(image, mask, seed=s, pipeline=pipeline)
        outputs.append(res["result"])
    return outputs

## § 7 · Evaluation

In [38]:
# ── evaluate_coco_quantitative ────────────────────────────────────────────────
# Runs all three pipelines on COCO subset × 3 seeds.
# Checkpoints after every sample; auto-resumes from partial CSV on restart.

def evaluate_coco_quantitative(
    subset: list[dict] | None = None,
    seeds:  list[int]  = SEEDS,
    out_csv: Path       = QUANT_CSV,
) -> None:
    if subset is None:
        if INDICES_FILE.exists():
            print("Loading cached subset indices…")
            with open(INDICES_FILE) as f:
                cached = json.load(f)
            print(f"Re-using {len(cached)} cached indices. "
                  "Re-run load_coco_subset() to change the selection.")
            # We still need the actual images/masks — reload from COCO
            subset = load_coco_subset()
        else:
            subset = load_coco_subset()
            save_subset_indices(subset)

    done = load_done_set(out_csv)
    pipelines = ["baseline", "vanilla", "improved"]
    total = len(subset) * len(seeds) * len(pipelines)
    bar = tqdm(total=total, desc="COCO eval")

    for sample in subset:
        image, mask, iid, cat = (
            sample["image"], sample["mask"],
            str(sample["image_id"]), sample["category"]
        )
        # Pre-compute caption once per sample (model-agnostic)
        prompt = caption_background(image, mask)

        for seed in seeds:
            for pipe_name in pipelines:
                key = (iid, str(seed), pipe_name)
                bar.update(1)
                if key in done:
                    continue

                set_all_seeds(seed)
                gen = make_generator(seed)
                t0  = time.perf_counter()
                try:
                    if pipe_name == "baseline":
                        result = sd_inpaint_pipe(
                            prompt=prompt, image=image, mask_image=mask, generator=gen
                        ).images[0]
                    elif pipe_name == "vanilla":
                        result = vanilla_pipe(
                            prompt=prompt, image=image, mask_image=mask, generator=gen
                        ).images[0]
                    else:
                        result = improved_pipe(
                            prompt=prompt, image=image, mask_image=mask, generator=gen
                        ).images[0]
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        torch.cuda.empty_cache(); gc.collect()
                        print(f"OOM on {iid}/{seed}/{pipe_name} — skipping")
                        continue
                    raise

                rt = time.perf_counter() - t0
                lp  = compute_lpips(result, image)
                mlp = compute_mask_lpips(result, image, mask)
                cs  = compute_clip_score(result, prompt)

                row = {
                    "image_id": iid, "category": cat, "seed": seed,
                    "pipeline": pipe_name,
                    "lpips": round(lp, 5), "mask_lpips": round(mlp, 5),
                    "clip_score": round(cs, 5), "runtime_sec": round(rt, 2),
                    "caption": prompt,
                }
                checkpoint_append([row], out_csv)
                done.add(key)

    bar.close()
    print(f"Quantitative eval complete. Results → {out_csv}")

In [39]:
# ── evaluate_ablation ─────────────────────────────────────────────────────────
# Disables one refinement at a time on a 30-sample subset, 1 seed.

def evaluate_ablation(
    subset:  list[dict] | None = None,
    n:       int  = ABLATION_SUBSET,
    seed:    int  = 42,
    out_csv: Path = ABLATION_CSV,
) -> None:
    if subset is None:
        subset = load_coco_subset()

    ablation_subset = subset[:n]
    done = load_done_set(out_csv)
    bar  = tqdm(total=n * len(ABLATION_KNOBS), desc="Ablation")

    for sample in ablation_subset:
        image, mask, iid, cat = (
            sample["image"], sample["mask"],
            str(sample["image_id"]), sample["category"]
        )
        prompt = caption_background(image, mask)

        for ablation_name, knobs in ABLATION_KNOBS.items():
            key = (iid, str(seed), ablation_name)
            bar.update(1)
            if key in done:
                continue

            set_all_seeds(seed)
            gen = make_generator(seed)
            neg = "" if ablation_name == "No Negative Prompt" else None
            try:
                result = improved_pipe(
                    prompt=prompt, image=image, mask_image=mask,
                    generator=gen, negative_prompt=neg, **knobs
                ).images[0]
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    torch.cuda.empty_cache(); gc.collect(); continue
                raise

            row = {
                "image_id": iid, "category": cat, "seed": seed,
                "pipeline": ablation_name,
                "lpips": round(compute_lpips(result, image), 5),
                "mask_lpips": round(compute_mask_lpips(result, image, mask), 5),
                "clip_score": round(compute_clip_score(result, prompt), 5),
                "runtime_sec": 0, "caption": prompt,
            }
            checkpoint_append([row], out_csv)
            done.add(key)

    bar.close()
    print(f"Ablation complete → {out_csv}")

In [40]:
# ── evaluate_qualitative ──────────────────────────────────────────────────────
# Generates side-by-side comparison grids for the first N samples.

def evaluate_qualitative(
    subset: list[dict],
    n: int = 10,
    out_dir: Path = QUAL_DIR,
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    for i, sample in enumerate(subset[:n]):
        image, mask, iid = sample["image"], sample["mask"], sample["image_id"]
        prompt = caption_background(image, mask)
        results = {}
        for pipe_name, pipe in [("baseline", sd_inpaint_pipe),
                                 ("vanilla",  vanilla_pipe),
                                 ("improved", improved_pipe)]:
            set_all_seeds(SEEDS[0])
            gen = make_generator(SEEDS[0])
            kw  = dict(prompt=prompt, image=image, mask_image=mask, generator=gen)
            results[pipe_name] = pipe(**kw).images[0]

        grid = make_comparison_grid(
            image, mask, results,
            save_path=out_dir / f"{iid:05}_comparison.png" if isinstance(iid, int)
                      else out_dir / f"{iid}_comparison.png"
        )
        display(grid)
        print(f"[{i+1}/{min(n, len(subset))}] prompt: {prompt}")

In [41]:
# ── build_user_study_pairs ────────────────────────────────────────────────────
# Generates 30 anonymized A/B pairs for 2AFC user study (Google Forms).

def build_user_study_pairs(
    subset: list[dict],
    n_pairs: int = 30,
    seed: int = 42,
    out_dir: Path = RESULTS_DIR / "user_study",
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    rng  = random.Random(seed)
    used = subset[:n_pairs]
    pairs = []

    for i, sample in enumerate(used):
        image, mask = sample["image"], sample["mask"]
        prompt      = caption_background(image, mask)
        iid         = sample["image_id"]

        # Generate improved and vanilla for comparison
        set_all_seeds(SEEDS[0])
        gen_a = make_generator(SEEDS[0])
        gen_b = make_generator(SEEDS[0])
        res_improved = improved_pipe(prompt=prompt, image=image, mask_image=mask, generator=gen_a).images[0]
        res_vanilla  = vanilla_pipe( prompt=prompt, image=image, mask_image=mask, generator=gen_b).images[0]

        # Randomize left/right
        flip    = rng.random() < 0.5
        img_a   = res_improved if not flip else res_vanilla
        img_b   = res_vanilla  if not flip else res_improved
        label_a = "improved" if not flip else "vanilla"

        # Save side-by-side
        combined = Image.new("RGB", (IMAGE_SIZE * 2 + 4, IMAGE_SIZE), (200, 200, 200))
        combined.paste(img_a, (0, 0))
        combined.paste(img_b, (IMAGE_SIZE + 4, 0))
        fname = out_dir / f"pair_{i+1:02d}.png"
        combined.save(str(fname))
        pairs.append({"pair": i + 1, "left": label_a, "image_id": str(iid), "fname": str(fname)})

    with open(out_dir / "pairs_key.json", "w") as f:
        json.dump(pairs, f, indent=2)
    print(f"Generated {len(pairs)} pairs in {out_dir}")
    print("Upload images to Google Drive, embed in Google Forms (2AFC: 'Which looks more realistic?')")

In [42]:
# ── Print results summary ─────────────────────────────────────────────────────

def print_results_summary(csv_path: Path = QUANT_CSV) -> None:
    if not csv_path.exists():
        print("No results CSV found. Run evaluate_coco_quantitative() first.")
        return
    from collections import defaultdict as dd
    totals: dict = dd(lambda: dd(lambda: {"lpips": 0., "mask_lpips": 0., "clip_score": 0., "n": 0}))
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            t = totals[row["pipeline"]]
            t["all"]["lpips"]      += float(row["lpips"])
            t["all"]["mask_lpips"] += float(row["mask_lpips"])
            t["all"]["clip_score"] += float(row["clip_score"])
            t["all"]["n"]          += 1
    print(f"{'Pipeline':<30}  {'LPIPS↓':>8}  {'MaskLPIPS↓':>12}  {'CLIP↑':>8}  {'N':>5}")
    print("-" * 70)
    for pipe, d in sorted(totals.items()):
        t, n = d["all"], d["all"]["n"]
        if n == 0: continue
        print(f"{pipe:<30}  {t['lpips']/n:8.4f}  {t['mask_lpips']/n:12.4f}  {t['clip_score']/n:8.4f}  {n:5}")

### Interactive Demo Widget
Click an image to remove an object. Two implementations: matplotlib interactive (primary) and ipywidgets fallback.

In [43]:
# ── Interactive demo (primary — matplotlib widget) ────────────────────────────
# Requires: %matplotlib widget (ipympl). Test on Day 4.
%matplotlib widget

def interactive_demo(image_path: str) -> None:
    image = Image.open(image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(np.array(image)); axes[0].set_title("Click to remove object"); axes[0].axis("off")
    axes[1].set_title("SAM Mask"); axes[1].axis("off")
    axes[2].set_title("Result"); axes[2].axis("off")
    plt.tight_layout()

    def on_click(event):
        if event.inaxes != axes[0]: return
        x, y = int(event.xdata), int(event.ydata)
        print(f"Clicked ({x}, {y}) — running pipeline…")
        out = run_pipeline(image, (x, y), mode="demo", seed=42, verbose=True)
        axes[1].clear(); axes[1].imshow(np.array(out["mask"]), cmap="gray"); axes[1].axis("off")
        axes[2].clear(); axes[2].imshow(np.array(out["result"])); axes[2].axis("off")
        axes[2].set_xlabel(f'Prompt: {out["prompt"][:60]}', fontsize=7)
        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("button_press_event", on_click)
    plt.show()

In [44]:
# ── Interactive demo (fallback — ipywidgets text input) ──────────────────────
# Use if %matplotlib widget fails to register clicks in Colab.
%matplotlib inline

def interactive_demo_fallback(image_path: str) -> None:
    image = Image.open(image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(np.array(image)); ax.set_title("Reference image (enter click coordinates below)"); ax.axis("off")
    plt.show()

    coord_input = widgets.Text(placeholder="x,y  e.g. 240,180", description="Click at:")
    pipeline_dd = widgets.Dropdown(options=["improved", "vanilla", "baseline"], value="improved", description="Pipeline:")
    run_btn     = widgets.Button(description="Remove object", button_style="primary")
    out_widget  = widgets.Output()

    def on_run(_):
        with out_widget:
            clear_output(wait=True)
            try:
                xc, yc = [int(v.strip()) for v in coord_input.value.split(",")]
            except ValueError:
                print("Enter coordinates as: x,y"); return
            print(f"Running {pipeline_dd.value} pipeline at ({xc}, {yc})…")
            res = run_pipeline(image, (xc, yc), mode="demo", seed=42,
                               pipeline=pipeline_dd.value, verbose=True)
            fig2, axes = plt.subplots(1, 3, figsize=(15, 5))
            axes[0].imshow(np.array(image)); axes[0].set_title(f"Input (click {xc},{yc})"); axes[0].axis("off")
            axes[1].imshow(np.array(res["mask"]), cmap="gray"); axes[1].set_title("Mask"); axes[1].axis("off")
            axes[2].imshow(np.array(res["result"])); axes[2].set_title("Result"); axes[2].axis("off")
            axes[2].set_xlabel(f'Prompt: {res["prompt"][:70]}', fontsize=7)
            plt.tight_layout(); plt.show()
            print(f"Runtime: {res['times']['total_sec']}s")

    run_btn.on_click(on_run)
    display(widgets.VBox([coord_input, pipeline_dd, run_btn, out_widget]))

---
### Run the pipeline
Uncomment the cells below to execute the full evaluation pipeline.

In [45]:
# ── COCO subset selection (run once, cached in results/) ──────────────────────
coco_subset = load_coco_subset()
save_subset_indices(coco_subset)

Loading shunk031/MSCOCO 2017 val (streaming)…


FileNotFoundError: Couldn't find file at https://huggingface.co/datasets/shunk031/MSCOCO/resolve/main/{'train': 'http://images.cocodataset.org/zips/train2017.zip', 'validation': 'http://images.cocodataset.org/zips/val2017.zip', 'test': 'http://images.cocodataset.org/zips/test2017.zip', 'unlabeled': 'http://images.cocodataset.org/zips/unlabeled2017.zip'}

In [ ]:
# ── Quantitative evaluation (~5h on T4 — run overnight) ──────────────────────
# evaluate_coco_quantitative(coco_subset)

In [ ]:
# ── Ablation study (~3h on T4) ────────────────────────────────────────────────
# evaluate_ablation(coco_subset)

In [ ]:
# ── Qualitative grids (first 10 samples) ─────────────────────────────────────
# evaluate_qualitative(coco_subset, n=10)

In [ ]:
# ── User study pairs ──────────────────────────────────────────────────────────
# build_user_study_pairs(coco_subset)

In [ ]:
# ── Results summary ───────────────────────────────────────────────────────────
# print_results_summary()

In [ ]:
# ── Interactive demo (uncomment one) ─────────────────────────────────────────
# interactive_demo("/content/drive/MyDrive/cv_project/demo_images/sample.jpg")
# interactive_demo_fallback("/content/drive/MyDrive/cv_project/demo_images/sample.jpg")